# Training Optimization and Model Selection - Dzifa, Avi, Erin

Task #2, part 2. Uses the pipeline from `Model_Setup_InitialTraining.ipynb` for full training runs:

1. GPU environment check
2. Tokenizers for RoBERTa and Legal-BERT (`nlpaueb/legal-bert-base-uncased`)
3. Training with the shared setup (41 labels, `BCEWithLogitsLoss`, Hugging Face `Trainer`)
4. Training and validation loss after every epoch
5. Best epoch and model chosen on the validation set
6. Best checkpoint, test predictions for Task #4, and comparison with the TF-IDF baseline

**Run modes.** `RUN_MODE = "auto"` runs **full** on a GPU and **smoke** on CPU. Smoke mode uses a small subset to check the pipeline; its scores are meaningless.

**Changes from the initial pipeline:** GPU instead of `use_cpu=True`, validation every epoch, best checkpoint kept, fixed seed.

## 0. Google Colab setup
On Colab: clones the repo, installs `requirements.txt`, mounts Google Drive (results are saved there). Does nothing elsewhere.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Break-Through-Tech/Accenture-1H-contract-review-challenge.git"
REPO_BRANCH = "feat/multi-label-training-optimization"   # change to "main" once this is merged
SAVE_TO_DRIVE = True                          # Colab only: keep results/checkpoints on Google Drive

if IN_COLAB:
    repo_dir = "/content/Accenture-1H-contract-review-challenge"
    if not os.path.exists(repo_dir):
        cloned = subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, repo_dir]).returncode == 0
        if not cloned:   # branch not on GitHub (yet): main has the same data files
            print(f"Branch {REPO_BRANCH} not found, cloning main instead.")
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, repo_dir], check=True)
    os.chdir(f"{repo_dir}/notebooks")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "../requirements.txt"], check=True)
    if SAVE_TO_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
print("In Colab:", IN_COLAB, "| working dir:", os.getcwd())

Mounted at /content/drive
In Colab: True | working dir: /content/Accenture-1H-contract-review-challenge/notebooks


## 1. Setup

In [ ]:
import gc, json, re, shutil, time, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import transformers
from datasets import Dataset, DatasetDict, Features, Sequence, Value, load_from_disk, disable_progress_bars
from sklearn.metrics import average_precision_score, precision_recall_fscore_support
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          Trainer, TrainingArguments, set_seed)
from sklearn.metrics import f1_score, precision_recall_curve

transformers.logging.set_verbosity_error()   # hide the expected "newly initialized classifier" warning
disable_progress_bars()                       # keep tokenization output short
print("torch", torch.__version__, "| transformers", transformers.__version__)

torch 2.11.0+cu130 | transformers 4.57.6


All settings. Remove a model from `MODELS_TO_RUN` to train it in a separate session.

In [ ]:
RUN_MODE = "auto"      # "auto" (full if a GPU is available, else smoke), "full", or "smoke"
SEED = 42
DATA_DIR = "../data/processed"

MODELS = {   # short tag -> Hugging Face model name
    "roberta": "roberta-base",
    "legal-bert": "nlpaueb/legal-bert-base-uncased",
}
MODELS_TO_RUN = ["roberta", "legal-bert"]
SKIP_FINISHED = True   # don't retrain a model whose summary.json already exists

MAX_LENGTH = 512       # both models' input limit
OVERLAP = 128          # chunk overlap used in Subword_Tokenization_Chunking.ipynb
THRESHOLD = 0.5        # fixed here; per-category thresholds are tuned in Task #3
SELECT_METRIC = "macro_ap"   # validation metric used to choose the best epoch (see section 5)

FULL = dict(num_epochs=15, learning_rate=2e-5, effective_batch_size=16, max_length=MAX_LENGTH,
            warmup_ratio=0.1, weight_decay=0.01, subset=None)
SMOKE = dict(num_epochs=3, learning_rate=5e-5, effective_batch_size=16, max_length=128,
             warmup_ratio=0.1, weight_decay=0.01, subset={"train": 384, "validation": 128, "test": 128})

if not os.path.exists(DATA_DIR):
    raise FileNotFoundError(f"{DATA_DIR} not found. Run this notebook from the notebooks/ folder.")
set_seed(SEED)

## 2. GPU training environment
Checks the device and turns on mixed precision on GPU (bf16 if supported, else fp16). The per-step batch size depends on GPU memory; gradient accumulation keeps the **effective batch size at 16**.

In [ ]:
HAS_GPU = torch.cuda.is_available()
MODE = ("full" if HAS_GPU else "smoke") if RUN_MODE == "auto" else RUN_MODE
CFG = dict(FULL if MODE == "full" else SMOKE)

print("Python:", platform.python_version(), "| CPU cores:", os.cpu_count())
if HAS_GPU:
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1e9
    print(f"GPU: {props.name} | {vram_gb:.1f} GB | CUDA {torch.version.cuda}")
    USE_BF16 = torch.cuda.is_bf16_supported()
    USE_FP16 = not USE_BF16
    per_device = 16 if vram_gb >= 40 else 8 if vram_gb >= 14 else 4
else:
    print("GPU: none found. Training runs on CPU.")
    if IN_COLAB:
        print("  -> On Colab: Runtime > Change runtime type > T4 GPU (or better), then run all again.")
    USE_BF16 = USE_FP16 = False
    per_device = 8
if MODE == "full" and not HAS_GPU:
    print("WARNING: full mode on CPU will take many hours per epoch.")

# Make sure the device can actually run a computation
device = torch.device("cuda" if HAS_GPU else "cpu")
a = torch.randn(1024, 1024, device=device)
assert torch.isfinite(a @ a).all()

CFG["per_device_batch_size"] = min(per_device, CFG["effective_batch_size"])
CFG["grad_accum_steps"] = CFG["effective_batch_size"] // CFG["per_device_batch_size"]
print(f"Device check passed on {device}. Mode: {MODE} | precision: "
      f"{'bf16' if USE_BF16 else 'fp16' if USE_FP16 else 'fp32'} | batch {CFG['per_device_batch_size']} "
      f"x {CFG['grad_accum_steps']} accumulation = {CFG['effective_batch_size']}")

Python: 3.13.16 | CPU cores: 12
GPU: NVIDIA A100-SXM4-40GB | 42.4 GB | CUDA 13.0
Device check passed on cuda. Mode: full | precision: bf16 | batch 16 x 1 accumulation = 16


In [ ]:
# results/ is committed, models/ is not. Each run gets its own folder (e.g. 15ep). Smoke runs stay under models/.
RUN_NAME = f"{CFG['num_epochs']}ep"
OUTPUT_ROOT = "/content/drive/MyDrive/Accenture-1H-contract-review-challenge" if (IN_COLAB and SAVE_TO_DRIVE) else ".."
RESULTS_DIR = f"{OUTPUT_ROOT}/results/training/{RUN_NAME}" if MODE == "full" else f"{OUTPUT_ROOT}/models/training_smoke/results"
MODELS_DIR = f"{OUTPUT_ROOT}/models/training/{RUN_NAME}" if MODE == "full" else f"{OUTPUT_ROOT}/models/training_smoke"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
print("Results:", RESULTS_DIR, "\nCheckpoints:", MODELS_DIR)

Results: /content/drive/MyDrive/Accenture-1H-contract-review-challenge/results/training/15ep 
Checkpoints: /content/drive/MyDrive/Accenture-1H-contract-review-challenge/models/training/15ep


## 3. Load the chunks and the validation split
Same inputs and 61-contract validation split (`val_titles.json`) as `Transformer_Dataset_Prep.ipynb`. The splits are rebuilt from the text because Legal-BERT needs its own tokenizer; `roberta_dataset/` only has RoBERTa token IDs.

In [ ]:
train_df = pd.read_parquet(f"{DATA_DIR}/train_chunks_labeled.parquet")
test_df = pd.read_parquet(f"{DATA_DIR}/test_chunks_labeled.parquet")
with open(f"{DATA_DIR}/categories.json") as f:
    CATEGORIES = json.load(f)
with open(f"{DATA_DIR}/val_titles.json") as f:
    val_titles = set(json.load(f)["val_titles"])
train_titles = set(train_df["title"]) - val_titles

FEATURES = Features({"title": Value("string"), "chunk_id": Value("int64"), "chunk_text": Value("string"),
                     "labels": Sequence(Value("float32"), length=len(CATEGORIES))})

def build_split(df, titles):
    part = df[df["title"].isin(titles)].reset_index(drop=True)
    out = part[["title", "chunk_id", "chunk_text"]].copy()
    out["labels"] = list(part[CATEGORIES].to_numpy(dtype=np.float32))
    return Dataset.from_pandas(out, features=FEATURES, preserve_index=False)

raw = DatasetDict({"train": build_split(train_df, train_titles),
                   "validation": build_split(train_df, val_titles),
                   "test": build_split(test_df, set(test_df["title"]))})

assert len(CATEGORIES) == 41 and len(val_titles) == 61 and len(train_titles) == 347
assert not (val_titles & train_titles) and not (set(train_df["title"]) & set(test_df["title"]))
assert (raw["train"].num_rows, raw["validation"].num_rows, raw["test"].num_rows) == (12112, 2069, 2898)
print({k: v.num_rows for k, v in raw.items()}, "| load checks passed.")

{'train': 12112, 'validation': 2069, 'test': 2898} | load checks passed.


## 4. Tokenizers: RoBERTa and Legal-BERT
- **RoBERTa** (`roberta-base`): cased byte-level BPE. Chunks were cut to 512 RoBERTa tokens.
- **Legal-BERT**: uncased WordPiece pretrained on legal text, so token counts differ.

**Punctuation cleanup for Legal-BERT.** Legal-BERT splits table borders (`--------`) into one token per dash, so some chunks exceed 1,700 tokens. Runs of 3+ identical punctuation marks are shortened to 3 before Legal-BERT tokenization. RoBERTa gets the original text. Apply the same cleanup at prediction time (noted in `training_info.json`).

In [ ]:
PUNCT_RUN = re.compile(r"([^\w\s])\1{2,}")
TEXT_CLEANUP = {"roberta": False, "legal-bert": True}

def clean_text(text):
    return PUNCT_RUN.sub(r"\1\1\1", text)

tokenizers = {tag: AutoTokenizer.from_pretrained(MODELS[tag], use_fast=True) for tag in MODELS}

def tokenize_split(ds, tag, max_length):
    tok, cleanup = tokenizers[tag], TEXT_CLEANUP[tag]
    def fn(batch):
        texts = [clean_text(t) for t in batch["chunk_text"]] if cleanup else batch["chunk_text"]
        return tok(texts, max_length=max_length, truncation=True)
    return ds.map(fn, batched=True, remove_columns=["chunk_text"])

for tag, tok in tokenizers.items():
    print(f"{tag:11s} {type(tok).__name__:20s} vocab {tok.vocab_size:,} | lowercase: {bool(getattr(tok, 'do_lower_case', False))} "
          f"| special: {tok.cls_token} ... {tok.sep_token}")

example = "This Agreement shall be governed by the laws of the State of New York."
for tag, tok in tokenizers.items():
    print(f"\n{tag}: {tok.tokenize(example)}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

roberta     RobertaTokenizerFast vocab 50,265 | lowercase: False | special: <s> ... </s>
legal-bert  BertTokenizerFast    vocab 30,522 | lowercase: True | special: [CLS] ... [SEP]

roberta: ['This', 'ĠAgreement', 'Ġshall', 'Ġbe', 'Ġgoverned', 'Ġby', 'Ġthe', 'Ġlaws', 'Ġof', 'Ġthe', 'ĠState', 'Ġof', 'ĠNew', 'ĠYork', '.']

legal-bert: ['this', 'agreement', 'shall', 'be', 'governed', 'by', 'the', 'laws', 'of', 'the', 'state', 'of', 'new', 'york', '.']


### How each tokenizer fits our chunks
Cuts of up to 128 tokens are safe because the next chunk repeats them (overlap). Cuts on a contract's last chunk, or longer than the overlap, lose text.

In [ ]:
def length_report(tag):
    tok, rows = tokenizers[tag], []
    for name, ds in raw.items():
        texts = [clean_text(t) for t in ds["chunk_text"]] if TEXT_CLEANUP[tag] else list(ds["chunk_text"])
        lengths = np.array([len(x) for x in tok(texts, verbose=False)["input_ids"]])
        dropped = np.maximum(lengths - MAX_LENGTH, 0)
        chunks = ds.select_columns(["title", "chunk_id"]).to_pandas()
        is_last = (chunks["chunk_id"] == chunks.groupby("title")["chunk_id"].transform("max")).to_numpy()
        rows.append({"model": tag, "split": name, "chunks": len(ds), "median_tokens": int(np.median(lengths)),
                     "max_tokens": int(lengths.max()), "truncated": int((dropped > 0).sum()),
                     "cut_past_overlap": int((dropped > OVERLAP).sum()), "last_chunk_cut": int((dropped[is_last] > 0).sum()),
                     "tokens_lost_%": round(100 * dropped.sum() / lengths.sum(), 2)})
    return rows

lengths_df = pd.DataFrame(length_report("roberta") + length_report("legal-bert"))
lengths_df

,model,split,chunks,median_tokens,max_tokens,truncated,cut_past_overlap,last_chunk_cut,tokens_lost_%
0,roberta,train,12112,512,515,1339,0,0,0.03
1,roberta,validation,2069,512,515,260,0,0,0.03
2,roberta,test,2898,512,515,379,0,0,0.03
3,legal-bert,train,12112,464,937,1016,27,4,0.47
4,legal-bert,validation,2069,470,708,162,3,2,0.28
5,legal-bert,test,2898,470,1228,223,4,1,0.41


In [ ]:
# RoBERTa: same as Transformer_Dataset_Prep.ipynb (at most 3 tokens cut, never a last chunk)
rob = lengths_df[lengths_df["model"] == "roberta"]
assert rob["max_tokens"].max() <= MAX_LENGTH + 3 and rob["last_chunk_cut"].sum() == 0
# Legal-BERT: the punctuation cleanup keeps the loss small
lb = lengths_df[lengths_df["model"] == "legal-bert"]
assert lb["tokens_lost_%"].max() < 1.0

# If roberta_dataset/ exists (built by Transformer_Dataset_Prep.ipynb), our RoBERTa tokens must match it exactly
roberta_full = DatasetDict({k: tokenize_split(v, "roberta", MAX_LENGTH) for k, v in raw.items()})
if os.path.exists(f"{DATA_DIR}/roberta_dataset"):
    prepared = load_from_disk(f"{DATA_DIR}/roberta_dataset")
    for k in raw:
        assert prepared[k]["title"] == roberta_full[k]["title"] and prepared[k]["chunk_id"] == roberta_full[k]["chunk_id"]
        assert prepared[k]["input_ids"] == roberta_full[k]["input_ids"] and prepared[k]["labels"] == roberta_full[k]["labels"]
    del prepared
    print("RoBERTa tokens match data/processed/roberta_dataset exactly.")
else:
    print("roberta_dataset/ not found (not in git); skipped the match check.")
del roberta_full
print("Tokenizer checks passed.")

roberta_dataset/ not found (not in git); skipped the match check.
Tokenizer checks passed.


**Result.** RoBERTa: about 11% of chunks lose 1–3 tokens, all within the overlap. Legal-BERT: most chunks are shorter, about 8% still exceed 512, and under 0.5% of tokens are lost after the cleanup.

## 5. Training setup
**Model and loss.** `AutoModelForSequenceClassification` with 41 outputs and `problem_type="multi_label_classification"` (`BCEWithLogitsLoss`). No class weights; those and per-category thresholds are Task #3.

**Best epoch = highest validation macro average precision (AP).** AP doesn't depend on a threshold, and macro averaging weights rare categories equally. Validation loss and F1 at 0.5 are reported too. Categories with no positives in a split are excluded (as in the baseline).

In [ ]:
def multilabel_scores(y_true, probs, threshold=THRESHOLD):
    y_true = np.asarray(y_true).astype(int)
    has_pos = y_true.sum(axis=0) > 0
    yt, pr = y_true[:, has_pos], probs[:, has_pos]
    yp = (pr >= threshold).astype(int)
    scores = {"macro_ap": average_precision_score(yt, pr, average="macro"),
              "micro_ap": average_precision_score(yt, pr, average="micro")}
    for avg in ["micro", "macro"]:
        p, r, f, _ = precision_recall_fscore_support(yt, yp, average=avg, zero_division=0)
        scores.update({f"{avg}_precision": p, f"{avg}_recall": r, f"{avg}_f1": f})
    scores["categories_scored"] = int(has_pos.sum())
    return scores

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return multilabel_scores(labels, sigmoid(logits))

# Quick check on made-up data: perfect predictions score 1, and a category with no positives is skipped
y = np.array([[1, 0, 0], [0, 1, 0], [1, 1, 0]])
s = multilabel_scores(y, y.astype(float))
assert s["macro_ap"] == 1 and s["micro_f1"] == 1 and s["categories_scored"] == 2
print("Metric checks passed.")

Metric checks passed.


In [ ]:
def prepare_splits(tag):
    # Tokenized train / validation / test for one model. Smoke mode uses a fixed random subset.
    splits = {}
    for name, ds in raw.items():
        if CFG["subset"]:
            ds = ds.shuffle(seed=SEED).select(range(CFG["subset"][name]))
        splits[name] = tokenize_split(ds, tag, CFG["max_length"])
    return splits

def training_args(out_dir):
    return TrainingArguments(
        output_dir=out_dir,
        num_train_epochs=CFG["num_epochs"],
        learning_rate=CFG["learning_rate"],
        per_device_train_batch_size=CFG["per_device_batch_size"],
        per_device_eval_batch_size=CFG["per_device_batch_size"] * 2,
        gradient_accumulation_steps=CFG["grad_accum_steps"],
        warmup_ratio=CFG["warmup_ratio"],
        weight_decay=CFG["weight_decay"],
        eval_strategy="epoch",            # validation loss + scores after every epoch
        logging_strategy="epoch",         # training loss averaged over each epoch
        save_strategy="epoch",
        save_total_limit=1,               # keeps the best checkpoint (and the latest one)
        save_only_model=True,             # no optimizer state: ~0.5 GB per checkpoint instead of ~1.5 GB
        load_best_model_at_end=True,
        metric_for_best_model=SELECT_METRIC,
        greater_is_better=True,
        bf16=USE_BF16, fp16=USE_FP16,
        dataloader_num_workers=2 if HAS_GPU else 0,
        dataloader_pin_memory=HAS_GPU,
        seed=SEED, data_seed=SEED,
        report_to="none",
        disable_tqdm=not HAS_GPU,         # keep the saved CPU smoke run readable
    )

## 6. Train
For each model: train for `num_epochs` with validation after each epoch, reload the best epoch, then save:
- checkpoint and tokenizer to `models/training/<run>/<model>/best/`
- history, test predictions and summary to `results/training/<run>/<model>/`

`<run>` is the epoch count (e.g. `15ep`), so new runs don't overwrite old ones. With `SKIP_FINISHED`, finished models are skipped.

In [ ]:
def epoch_history(log_history):
    train = {round(h["epoch"]): h["loss"] for h in log_history if "loss" in h and "eval_loss" not in h}
    rows = []
    for h in log_history:
        if "eval_loss" in h:
            e = round(h["epoch"])
            rows.append({"epoch": e, "train_loss": train.get(e), "val_loss": h["eval_loss"],
                         **{k[5:]: v for k, v in h.items() if k.startswith("eval_") and k[5:] in
                            ["macro_ap", "micro_ap", "micro_f1", "macro_f1", "micro_precision", "micro_recall"]}})
    return pd.DataFrame(rows)


In [ ]:

def run_experiment(tag):
    res_dir, model_dir = f"{RESULTS_DIR}/{tag}", f"{MODELS_DIR}/{tag}"
    if SKIP_FINISHED and os.path.exists(f"{res_dir}/summary.json"):
        print(f"[{tag}] already finished, skipping (set SKIP_FINISHED = False to retrain)")
        return
    os.makedirs(res_dir, exist_ok=True)
    set_seed(SEED)
    splits = prepare_splits(tag)
    cols = ["input_ids", "attention_mask", "labels"]
    model = AutoModelForSequenceClassification.from_pretrained(
        MODELS[tag], num_labels=len(CATEGORIES), problem_type="multi_label_classification",
        id2label=dict(enumerate(CATEGORIES)), label2id={c: i for i, c in enumerate(CATEGORIES)})
    trainer = Trainer(model=model, args=training_args(f"{model_dir}/checkpoints"),
                      train_dataset=splits["train"].select_columns(cols),
                      eval_dataset=splits["validation"].select_columns(cols),
                      data_collator=DataCollatorWithPadding(tokenizers[tag]),
                      compute_metrics=compute_metrics)

    print(f"[{tag}] training {MODELS[tag]} on {splits['train'].num_rows:,} chunks for {CFG['num_epochs']} epochs ...")
    start = time.time()
    trainer.train()
    minutes = (time.time() - start) / 60

    history = epoch_history(trainer.state.log_history)
    best = history.loc[history[SELECT_METRIC].idxmax()]
    best_epoch = int(best["epoch"])
    history.to_csv(f"{res_dir}/epoch_history.csv", index=False)

    # Best checkpoint (load_best_model_at_end already put it back into trainer.model)
    best_dir = f"{model_dir}/best"
    trainer.save_model(best_dir)
    tokenizers[tag].save_pretrained(best_dir)

    # Per-category validation scores of the best epoch (input for Task #3)
    val_probs = sigmoid(trainer.predict(splits["validation"].select_columns(cols)).predictions)
    y_val = np.array(splits["validation"]["labels"])
    pd.DataFrame({"val_ap": [average_precision_score(y_val[:, j], val_probs[:, j]) if y_val[:, j].any() else np.nan
                             for j in range(len(CATEGORIES))],
                  "val_positives": y_val.sum(0).astype(int)}, index=CATEGORIES).to_csv(f"{res_dir}/val_per_category.csv")

    # Test predictions for Task #4: one row per chunk, one probability per category
    test_probs = sigmoid(trainer.predict(splits["test"].select_columns(cols)).predictions)
    preds = pd.DataFrame(test_probs, columns=CATEGORIES)
    preds.insert(0, "chunk_id", splits["test"]["chunk_id"])
    preds.insert(0, "title", splits["test"]["title"])
    preds.to_parquet(f"{res_dir}/test_predictions.parquet", index=False)
    test_scores = multilabel_scores(np.array(splits["test"]["labels"]), test_probs)

    summary = {"model": MODELS[tag], "tag": tag, "mode": MODE, "best_epoch": best_epoch,
               "select_metric": SELECT_METRIC, "best_val": {k: float(v) for k, v in best.items() if k != "epoch"},
               "test_at_best_epoch": {k: float(v) for k, v in test_scores.items()},
               "train_minutes": round(minutes, 1),
               "device": torch.cuda.get_device_name(0) if HAS_GPU else "cpu",
               "config": CFG, "text_cleanup": TEXT_CLEANUP[tag],
               "train_chunks": splits["train"].num_rows, "checkpoint": best_dir}
    with open(f"{best_dir}/training_info.json", "w") as f:
        json.dump({**summary, "categories": CATEGORIES,
                   "note": "apply clean_text() (runs of 3+ identical punctuation -> 3) before tokenizing" if TEXT_CLEANUP[tag] else ""}, f, indent=1)
    with open(f"{res_dir}/summary.json", "w") as f:
        json.dump(summary, f, indent=1)

    print(f"[{tag}] done in {minutes:.1f} min. Best epoch: {best_epoch} (val {SELECT_METRIC} {best[SELECT_METRIC]:.3f})")
    del trainer, model
    gc.collect()
    if HAS_GPU:
        torch.cuda.empty_cache()

for tag in MODELS_TO_RUN:
    run_experiment(tag)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

[roberta] training roberta-base on 12,112 chunks for 15 epochs ...


Epoch,Training Loss,Validation Loss,Macro Ap,Micro Ap,Micro Precision,Micro Recall,Micro F1,Macro Precision,Macro Recall,Macro F1,Categories Scored
1,0.262300,0.098909,0.033436,0.039080,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,41
2,0.084800,0.078137,0.173259,0.297322,0.824885,0.105480,0.187043,0.106565,0.063171,0.067748,41
3,0.063300,0.062247,0.346593,0.514486,0.878193,0.263406,0.405258,0.195376,0.136318,0.153051,41
4,0.048900,0.051314,0.407149,0.606987,0.807256,0.419564,0.552152,0.345919,0.251691,0.268256,41
5,0.040400,0.046978,0.452483,0.647565,0.806580,0.476724,0.599259,0.437535,0.303228,0.326105,41
6,0.034500,0.045103,0.472297,0.665292,0.757728,0.534473,0.626814,0.464833,0.346381,0.367139,41
7,0.029900,0.045386,0.502695,0.681823,0.704082,0.609900,0.653615,0.473993,0.416230,0.421522,41
8,0.026200,0.042608,0.512782,0.692129,0.748886,0.593989,0.662504,0.555020,0.406013,0.437527,41
9,0.023000,0.041720,0.524368,0.698993,0.752743,0.606364,0.671671,0.582398,0.427425,0.468069,41
10,0.020400,0.042134,0.535197,0.697649,0.746143,0.626989,0.681396,0.574094,0.444989,0.479176,41


[roberta] done in 13.5 min. Best epoch: 14 (val macro_ap 0.560)


pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

[legal-bert] training nlpaueb/legal-bert-base-uncased on 12,112 chunks for 15 epochs ...


Epoch,Training Loss,Validation Loss,Macro Ap,Micro Ap,Micro Precision,Micro Recall,Micro F1,Macro Precision,Macro Recall,Macro F1,Categories Scored
1,0.298200,0.103924,0.029337,0.037608,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,41


KeyboardInterrupt: 

## 7. Compare epochs and models
### Loss per epoch

In [ ]:
summaries, histories = {}, {}
for tag in MODELS:
    if os.path.exists(f"{RESULTS_DIR}/{tag}/summary.json"):
        with open(f"{RESULTS_DIR}/{tag}/summary.json") as f:
            summaries[tag] = json.load(f)
        histories[tag] = pd.read_csv(f"{RESULTS_DIR}/{tag}/epoch_history.csv")
assert summaries, "No finished runs found. Run section 6 first."

INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e5e4e0"
COLORS = {"roberta": "#2a78d6", "legal-bert": "#eb6834"}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for tag, h in histories.items():
    axes[0].plot(h["epoch"], h["train_loss"], marker="o", color=COLORS[tag], label=f"{tag} train")
    axes[0].plot(h["epoch"], h["val_loss"], marker="o", linestyle="--", color=COLORS[tag], label=f"{tag} validation")
    axes[1].plot(h["epoch"], h[SELECT_METRIC], marker="o", color=COLORS[tag], label=tag)
    b = summaries[tag]["best_epoch"]
    axes[1].scatter([b], [h.loc[h["epoch"] == b, SELECT_METRIC].iloc[0]], s=160, facecolors="none",
                    edgecolors=COLORS[tag], linewidths=2, zorder=3)
axes[0].set_title("Training vs validation loss", loc="left", color=INK)
axes[0].set_ylabel("BCE loss")
axes[1].set_title(f"Validation {SELECT_METRIC} (circle = best epoch)", loc="left", color=INK)
axes[1].set_ylabel(SELECT_METRIC)
for ax in axes:
    ax.set_xlabel("Epoch")
    ax.set_xticks(sorted({e for h in histories.values() for e in h["epoch"]}))
    ax.grid(axis="y", color=GRID)
    ax.set_axisbelow(True)
    ax.legend(frameon=False, fontsize=9)
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
    ax.tick_params(colors=MUTED)
if MODE == "smoke":
    fig.suptitle("SMOKE RUN (CPU, tiny subset): checks the pipeline only, the numbers are not meaningful", color="#b3261e", fontsize=10)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()

### Validation scores per epoch

In [ ]:
table = pd.concat({tag: h.set_index("epoch") for tag, h in histories.items()}, names=["model", "epoch"])
table.round(4)

### Best model and epoch
Chosen on validation only. Test is scored once for comparison with TF-IDF (`TF-IDF_Baseline.ipynb`: micro F1 0.591, macro F1 0.517 at 0.5).

In [ ]:
TFIDF_TEST = {"micro_f1": 0.591, "macro_f1": 0.517}   # from TF-IDF_Baseline.ipynb

best_tag = max(summaries, key=lambda t: summaries[t]["best_val"][SELECT_METRIC])
rows = []
for tag, s in summaries.items():
    rows.append({"model": tag, "best_epoch": s["best_epoch"], f"val_{SELECT_METRIC}": s["best_val"][SELECT_METRIC],
                 "val_micro_f1": s["best_val"]["micro_f1"], "val_macro_f1": s["best_val"]["macro_f1"],
                 "test_micro_f1": s["test_at_best_epoch"]["micro_f1"], "test_macro_f1": s["test_at_best_epoch"]["macro_f1"],
                 "train_minutes": s["train_minutes"], "chosen": tag == best_tag})
rows.append({"model": "tf-idf baseline", "test_micro_f1": TFIDF_TEST["micro_f1"], "test_macro_f1": TFIDF_TEST["macro_f1"]})
comparison = pd.DataFrame(rows).set_index("model")
comparison.to_csv(f"{RESULTS_DIR}/model_comparison.csv")

print(f"Chosen: {best_tag} ({summaries[best_tag]['model']}), epoch {summaries[best_tag]['best_epoch']}, "
      f"checkpoint: {summaries[best_tag]['checkpoint']}")
if MODE == "smoke":
    print("SMOKE RUN: the comparison only shows the pipeline works. Re-run on a GPU for real results.")
comparison.round(3)

### Weakest categories on validation (chosen model)
Focus areas for Task #3.

In [ ]:
per_cat = pd.read_csv(f"{RESULTS_DIR}/{best_tag}/val_per_category.csv", index_col=0)
per_cat.sort_values("val_ap").head(10).round(3)

## 8. Files saved
| File | In git | Contents |
|---|---|---|
| `results/training/<run>/<model>/epoch_history.csv` | Yes | Loss and validation scores per epoch |
| `results/training/<run>/<model>/summary.json` | Yes | Best epoch, validation and test scores, settings |
| `results/training/<run>/<model>/val_per_category.csv` | Yes | Validation AP per category (for Task #3) |
| `results/training/<run>/<model>/test_predictions.parquet` | Yes | `title`, `chunk_id`, 41 probabilities per test chunk (for Task #4) |
| `results/training/<run>/model_comparison.csv`, `loss_curves.png` | Yes | Comparison (marks the chosen model) and loss plot |
| `models/training/<run>/<model>/best/` | No (~0.5 GB) | Best checkpoint, tokenizer, `training_info.json` |

On Colab these are under `MyDrive/Accenture-1H-contract-review-challenge/`. Smoke runs go to `models/training_smoke/`. The next cell downloads the results as a zip.

In [ ]:
if IN_COLAB and MODE == "full":
    archive = shutil.make_archive("/content/results_training", "zip", RESULTS_DIR)
    from google.colab import files
    files.download(archive)
    print("Downloading", archive, f"- unzip it into results/training/{RUN_NAME}/ in the repo and commit.")
else:
    print("Nothing to download here." if not IN_COLAB else "Smoke run: nothing worth downloading.")

## 9. Results and limitations
**Results (A100, 15 epochs).** Legal-BERT is chosen: validation macro AP 0.593 vs 0.542 for RoBERTa. At threshold 0.5 it reaches test micro F1 **0.703** and macro F1 **0.569**, vs 0.591 / 0.517 for TF-IDF. Validation loss bottoms out around epoch 11 and AP barely changes after, so 11–15 epochs is enough. Files are in `results/training/15ep/`.

**Limitations**
- Legal-BERT still loses under 0.5% of tokens to truncation; re-chunking with its tokenizer would fix this.
- Fixed 0.5 threshold and no class weights, so rare categories score low (Task #3).
- One seed and one learning rate: differences of about ±0.01 may be noise.
- TF-IDF trained on all 408 contracts; the transformers on 347 (61 held out for validation).
- Price Restrictions has no test positives, so it's excluded from test averages.

## 10. Decision Threshold for Legal Bert and Roberta



I will first get the best checkpoint of Legal-Bert

In [ ]:
legal_bert_checkpoint = f"{MODELS_DIR}/legal-bert/best"
legal_bert_threshold_model = AutoModelForSequenceClassification.from_pretrained(legal_bert_checkpoint)

OSError: Error no file named pytorch_model.bin, model.safetensors, tf_model.h5, model.ckpt.index or flax_model.msgpack found in directory /content/drive/MyDrive/Accenture-1H-contract-review-challenge/models/training/15ep/legal-bert/best.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(legal_bert_checkpoint)
threshold_args = training_args(f"{legal_bert_checkpoint}/threshold_eval")
threshold_args.eval_strategy = "no"
threshold_trainer = Trainer(
    model=legal_bert_threshold_model,
    args=threshold_args,
    data_collator=DataCollatorWithPadding(tokenizer),
)

I will now run the model on validation dataset to get predictions, then convert to probabilities.

In [ ]:
legal_bert_splits = prepare_splits("legal-bert")

val_output = threshold_trainer.predict(
    legal_bert_splits["validation"].select_columns(
        ["input_ids", "attention_mask", "labels"]
    )
)

val_probs = sigmoid(val_output.predictions)
val_labels = val_output.label_ids

print("Probabilities shape:", val_probs.shape)
print("Labels shape:", val_labels.shape)

Tuning the 41 thresholds now in the validation set


In [ ]:
def tuning_threshold(val_probs, val_labels):
  thresholds = []
  results = []

  for i, category in enumerate(CATEGORIES):
      y_true = val_labels[:, i].astype(int)
      y_prob = val_probs[:, i]

      # Some categories may not have both positive and negative examples
      if len(np.unique(y_true)) < 2:
          best_threshold = 0.5
          best_f1 = f1_score(
              y_true, (y_prob >= 0.5).astype(int), zero_division=0
          )
      else:
          precision, recall, candidates = precision_recall_curve(
              y_true, y_prob
          )

          f1_scores = (
              2 * precision[:-1] * recall[:-1]
              / (precision[:-1] + recall[:-1] + 1e-8)
          )

          best_idx = np.argmax(f1_scores)
          best_threshold = float(candidates[best_idx])
          best_f1 = float(f1_scores[best_idx])

      baseline_f1 = f1_score(
          y_true, (y_prob >= 0.5).astype(int), zero_division=0
      )

      thresholds.append(best_threshold)
      results.append({
          "category": category,
          "threshold": best_threshold,
          "f1_at_0.5": baseline_f1,
          "f1_at_tuned_threshold": best_f1,
          "positive_examples": int(y_true.sum())
      })

  threshold_results = pd.DataFrame(results)

  display(
      threshold_results.sort_values(
          "f1_at_tuned_threshold", ascending=True
      ).round(3)
  )
  return threshold_results


In [ ]:
threshold_results = tuning_threshold(val_probs, val_labels)

In [ ]:
print("Mean per-category F1 at 0.5:",
      threshold_results["f1_at_0.5"].mean().round(3))

print("Mean per-category F1 with tuned thresholds:",
      threshold_results["f1_at_tuned_threshold"].mean().round(3))

Now onto the test set where we see if f1 scores got higher

In [ ]:
test_output = threshold_trainer.predict(
    legal_bert_splits["test"].select_columns(
        ["input_ids", "attention_mask", "labels"]
    )
)
test_probs = sigmoid(test_output.predictions)
test_labels = test_output.label_ids.astype(int)

In [ ]:
print("Probabilities shape:", test_probs.shape)
print("Labels shape:", test_labels.shape)

We have to make sure the categories are in the same order

In [ ]:

tuned_thresholds = (
    threshold_results.set_index("category")
    .loc[CATEGORIES, "threshold"]
    .to_numpy()
)

Make predictions using both approaches of 0.5 vs tuned

In [ ]:

test_preds_05 = (test_probs >= 0.5).astype(int)
test_preds_tuned = (test_probs >= tuned_thresholds).astype(int)


In [ ]:
print("Test macro F1 at 0.5:",
      round(f1_score(test_labels, test_preds_05,
                     average="macro", zero_division=0), 3))

print("Test macro F1 with tuned thresholds:",
      round(f1_score(test_labels, test_preds_tuned,
                     average="macro", zero_division=0), 3))

print("Test micro F1 at 0.5:",
      round(f1_score(test_labels, test_preds_05,
                     average="micro", zero_division=0), 3))

print("Test micro F1 with tuned thresholds:",
      round(f1_score(test_labels, test_preds_tuned,
                     average="micro", zero_division=0), 3))

We see that by tuning the thresholds, it brought up macro F1 (0.555 -> 0.593) but it lowered the micro F1 (0.703 -> 0.575) for Legal Bert.


Now I will so the same thing for Roberta

In [ ]:
roberta_checkpoint = f"{MODELS_DIR}/roberta/best"
roberta_threshold_model = AutoModelForSequenceClassification.from_pretrained(roberta_checkpoint)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(roberta_checkpoint)
threshold_args = training_args(f"{roberta_checkpoint}/threshold_eval")
threshold_args.eval_strategy = "no"
threshold_trainer = Trainer(
    model=roberta_threshold_model,
    args=threshold_args,
    data_collator=DataCollatorWithPadding(tokenizer),
)

In [ ]:
roberta_splits = prepare_splits("roberta")

val_output = threshold_trainer.predict(
    roberta_splits["validation"].select_columns(
        ["input_ids", "attention_mask", "labels"]
    )
)

val_probs = sigmoid(val_output.predictions)
val_labels = val_output.label_ids

print("Probabilities shape:", val_probs.shape)
print("Labels shape:", val_labels.shape)

In [ ]:
threshold_results = tuning_threshold(val_probs, val_labels)

tuned_thresholds = (
    threshold_results.set_index("category")
    .loc[CATEGORIES, "threshold"]
    .to_numpy()
)

In [ ]:
test_output = threshold_trainer.predict(
    roberta_splits["test"].select_columns(
        ["input_ids", "attention_mask", "labels"]
    )
)
test_probs = sigmoid(test_output.predictions)
test_labels = test_output.label_ids.astype(int)

In [ ]:
test_preds_05 = (test_probs >= 0.5).astype(int)
test_preds_tuned = (test_probs >= tuned_thresholds).astype(int)
print("Test macro F1 at 0.5:",
      round(f1_score(test_labels, test_preds_05,
                     average="macro", zero_division=0), 3))

print("Test macro F1 with tuned thresholds:",
      round(f1_score(test_labels, test_preds_tuned,
                     average="macro", zero_division=0), 3))

print("Test micro F1 at 0.5:",
      round(f1_score(test_labels, test_preds_05,
                     average="micro", zero_division=0), 3))

print("Test micro F1 with tuned thresholds:",
      round(f1_score(test_labels, test_preds_tuned,
                     average="micro", zero_division=0), 3))

We can observe a similar pattern with the roberta (Macro 0.51->0.541 and Micro 0.692->0.563). Tuned thresholds improved average performance across categories, but reduced overall micro F1. Will proceed to now focus on Legal Bert and do weighted loss experiments.

## 11. Retraining Legal Bert with Weighted Loss

In [ ]:
# Extract binary labels array from training split (shape: [num_chunks, 41])
train_labels = np.array(raw["train"]["labels"])
num_total = len(train_labels)
num_positives = train_labels.sum(axis=0)

# Calculate ratio (negatives / positives) with a safeguard against division by zero
num_negatives = num_total - num_positives
# pos_weights = num_negatives / np.maximum(num_positives, 1.0)
pos_weights = np.clip(np.sqrt(num_negatives / np.maximum(num_positives, 1.0)), 1.0, 10.0)

pos_weight_tensor = torch.tensor(pos_weights, dtype=torch.float32)
print("pos_weight min / median / max:",
      round(float(pos_weight_tensor.min()), 2),
      round(float(pos_weight_tensor.median()), 2),
      round(float(pos_weight_tensor.max()), 2))

pos_weight min / median / max: 4.48 8.31 10.0


In [ ]:
class WeightedTrainer(Trainer):
    def __init__(self, *args, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        # Ensure pos_weight is on the same device and precision as logits
        pos_weight = self.pos_weight.to(device=logits.device, dtype=logits.dtype)
        loss_fct = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

        loss = loss_fct(logits, labels.float())
        return (loss, outputs) if return_outputs else loss

In [ ]:

def run_experiment(tag):
    res_dir, model_dir = f"{RESULTS_DIR}/{tag}-weighted", f"{MODELS_DIR}/{tag}-weighted"
    if SKIP_FINISHED and os.path.exists(f"{res_dir}/summary.json"):
        print(f"[{tag}] already finished, skipping (set SKIP_FINISHED = False to retrain)")
        return
    os.makedirs(res_dir, exist_ok=True)
    set_seed(SEED)
    splits = prepare_splits(tag)
    cols = ["input_ids", "attention_mask", "labels"]
    model = AutoModelForSequenceClassification.from_pretrained(
        MODELS[tag], num_labels=len(CATEGORIES), problem_type="multi_label_classification",
        id2label=dict(enumerate(CATEGORIES)), label2id={c: i for i, c in enumerate(CATEGORIES)})
    trainer = WeightedTrainer(
        model=model,
        args=training_args(f"{model_dir}/checkpoints"),
        train_dataset=splits["train"].select_columns(cols),
        eval_dataset=splits["validation"].select_columns(cols),
        data_collator=DataCollatorWithPadding(tokenizers[tag]),
        compute_metrics=compute_metrics,
        pos_weight=pos_weight_tensor
    )

    print(f"[{tag}] training {MODELS[tag]} on {splits['train'].num_rows:,} chunks for {CFG['num_epochs']} epochs ...")
    start = time.time()
    trainer.train()
    minutes = (time.time() - start) / 60

    history = epoch_history(trainer.state.log_history)
    best = history.loc[history[SELECT_METRIC].idxmax()]
    best_epoch = int(best["epoch"])
    history.to_csv(f"{res_dir}/epoch_history.csv", index=False)

    # Best checkpoint (load_best_model_at_end already put it back into trainer.model)
    best_dir = f"{model_dir}/best"
    trainer.save_model(best_dir)
    tokenizers[tag].save_pretrained(best_dir)

    # Per-category validation scores of the best epoch (input for Task #3)
    val_probs = sigmoid(trainer.predict(splits["validation"].select_columns(cols)).predictions)
    y_val = np.array(splits["validation"]["labels"])
    pd.DataFrame({"val_ap": [average_precision_score(y_val[:, j], val_probs[:, j]) if y_val[:, j].any() else np.nan
                             for j in range(len(CATEGORIES))],
                  "val_positives": y_val.sum(0).astype(int)}, index=CATEGORIES).to_csv(f"{res_dir}/val_per_category.csv")

    # Test predictions for Task #4: one row per chunk, one probability per category
    test_probs = sigmoid(trainer.predict(splits["test"].select_columns(cols)).predictions)
    preds = pd.DataFrame(test_probs, columns=CATEGORIES)
    preds.insert(0, "chunk_id", splits["test"]["chunk_id"])
    preds.insert(0, "title", splits["test"]["title"])
    preds.to_parquet(f"{res_dir}/test_predictions.parquet", index=False)
    test_scores = multilabel_scores(np.array(splits["test"]["labels"]), test_probs)

    summary = {"model": MODELS[tag], "tag": tag, "mode": MODE, "best_epoch": best_epoch,
               "select_metric": SELECT_METRIC, "best_val": {k: float(v) for k, v in best.items() if k != "epoch"},
               "test_at_best_epoch": {k: float(v) for k, v in test_scores.items()},
               "train_minutes": round(minutes, 1),
               "device": torch.cuda.get_device_name(0) if HAS_GPU else "cpu",
               "config": CFG, "text_cleanup": TEXT_CLEANUP[tag],
               "train_chunks": splits["train"].num_rows, "checkpoint": best_dir}
    with open(f"{best_dir}/training_info.json", "w") as f:
        json.dump({**summary, "categories": CATEGORIES,
                   "note": "apply clean_text() (runs of 3+ identical punctuation -> 3) before tokenizing" if TEXT_CLEANUP[tag] else ""}, f, indent=1)
    with open(f"{res_dir}/summary.json", "w") as f:
        json.dump(summary, f, indent=1)

    print(f"[{tag}] done in {minutes:.1f} min. Best epoch: {best_epoch} (val {SELECT_METRIC} {best[SELECT_METRIC]:.3f})")
    del trainer, model
    gc.collect()
    if HAS_GPU:
        torch.cuda.empty_cache()

run_experiment("legal-bert")

[legal-bert] training nlpaueb/legal-bert-base-uncased on 12,112 chunks for 15 epochs ...


Epoch,Training Loss,Validation Loss,Macro Ap,Micro Ap,Micro Precision,Micro Recall,Micro F1,Macro Precision,Macro Recall,Macro F1,Categories Scored
1,0.476500,0.332365,0.217591,0.322675,0.645914,0.195639,0.300317,0.198414,0.119638,0.131887,41
2,0.231900,0.206369,0.471380,0.630676,0.458153,0.687095,0.549741,0.399994,0.530517,0.439559,41
3,0.155100,0.168173,0.530002,0.678766,0.444444,0.770772,0.563793,0.403732,0.629957,0.465558,41
4,0.116400,0.149593,0.582604,0.708401,0.479928,0.789039,0.596835,0.445434,0.668083,0.517046,41
5,0.089900,0.146576,0.589862,0.717396,0.498691,0.785504,0.610069,0.442697,0.687307,0.524264,41
6,0.071500,0.139747,0.610548,0.725951,0.531250,0.791397,0.635740,0.508273,0.688988,0.562076,41
7,0.058900,0.143899,0.610150,0.730259,0.561913,0.775486,0.651646,0.520512,0.664147,0.573638,41
8,0.048300,0.146722,0.618689,0.739154,0.600734,0.771361,0.675439,0.573579,0.667957,0.598603,41
9,0.040900,0.142288,0.627307,0.741549,0.565346,0.790218,0.659130,0.531237,0.686687,0.590939,41
10,0.035300,0.150494,0.630614,0.742110,0.616627,0.764879,0.682799,0.557795,0.656374,0.596042,41


[legal-bert] done in 13.5 min. Best epoch: 14 (val macro_ap 0.636)


In [ ]:
w_res, w_model = f"{RESULTS_DIR}/legal-bert-weighted", f"{MODELS_DIR}/legal-bert-weighted/best"
print(w_res, os.listdir(w_res)); print(w_model, os.listdir(w_model))

rows = {}
for name in ["legal-bert", "legal-bert-weighted"]:
    s = json.load(open(f"{RESULTS_DIR}/{name}/summary.json"))
    v, t = s["best_val"], s["test_at_best_epoch"]
    rows[name] = {"best_epoch": s["best_epoch"], "val_macro_ap": v["macro_ap"], "val_macro_f1": v["macro_f1"],
                  "val_micro_f1": v["micro_f1"], "test_macro_ap": t["macro_ap"],
                  "test_macro_f1": t["macro_f1"], "test_micro_f1": t["micro_f1"]}
pd.DataFrame(rows).T.round(3)

/content/drive/MyDrive/Accenture-1H-contract-review-challenge/results/training/15ep/legal-bert-weighted ['epoch_history.csv', 'val_per_category.csv', 'test_predictions.parquet', 'summary.json']
/content/drive/MyDrive/Accenture-1H-contract-review-challenge/models/training/15ep/legal-bert-weighted/best ['config.json', 'model.safetensors', 'tokenizer_config.json', 'special_tokens_map.json', 'vocab.txt', 'tokenizer.json', 'training_args.bin', 'training_info.json', 'threshold_eval']


,best_epoch,val_macro_ap,val_macro_f1,val_micro_f1,test_macro_ap,test_macro_f1,test_micro_f1
legal-bert,15.0,0.593,0.532,0.714,0.632,0.569,0.703
legal-bert-weighted,14.0,0.636,0.626,0.700,0.669,0.658,0.709


## 12. Decision Threshold for Weighted Legal Bert Model

In [ ]:
legal_bert_weighted_checkpoint = f"{MODELS_DIR}/legal-bert-weighted/best"
legal_bert_weighted_threshold_model = AutoModelForSequenceClassification.from_pretrained(legal_bert_weighted_checkpoint)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(legal_bert_weighted_checkpoint)
threshold_args = training_args(f"{legal_bert_weighted_checkpoint}/threshold_eval")
threshold_args.eval_strategy = "no"
threshold_trainer = Trainer(
    model=legal_bert_weighted_threshold_model,
    args=threshold_args,
    data_collator=DataCollatorWithPadding(tokenizer),
)

In [ ]:
legal_bert_weighted_splits = prepare_splits("legal-bert")

val_output = threshold_trainer.predict(
    legal_bert_weighted_splits["validation"].select_columns(
        ["input_ids", "attention_mask", "labels"]
    )
)

val_probs = sigmoid(val_output.predictions)
val_labels = val_output.label_ids

print("Probabilities shape:", val_probs.shape)
print("Labels shape:", val_labels.shape)

Probabilities shape: (2069, 41)
Labels shape: (2069, 41)


In [ ]:
threshold_results = tuning_threshold(val_probs, val_labels)

,category,threshold,f1_at_0.5,f1_at_tuned_threshold,positive_examples
1,Affiliate License-Licensor,0.021,0.000,0.019,3
30,Price Restrictions,0.682,0.200,0.222,8
7,Competitive Restriction Exception,0.215,0.190,0.273,14
21,Most Favored Nation,0.370,0.286,0.364,4
39,Volume Restriction,0.514,0.368,0.368,19
22,No-Solicit Of Customers,0.860,0.300,0.462,8
29,Post-Termination Services,0.529,0.465,0.469,70
0,Affiliate License-Licensee,0.714,0.475,0.523,25
17,Joint Ip Ownership,0.745,0.462,0.545,20
25,Non-Disparagement,0.548,0.500,0.556,8


In [ ]:
print("Mean per-category F1 at 0.5:",
      threshold_results["f1_at_0.5"].mean().round(3))

print("Mean per-category F1 with tuned thresholds:",
      threshold_results["f1_at_tuned_threshold"].mean().round(3))

Mean per-category F1 at 0.5: 0.626
Mean per-category F1 with tuned thresholds: 0.667


In [ ]:
test_output = threshold_trainer.predict(
    legal_bert_weighted_splits["test"].select_columns(
        ["input_ids", "attention_mask", "labels"]
    )
)
test_probs = sigmoid(test_output.predictions)
test_labels = test_output.label_ids.astype(int)

In [ ]:
print("Probabilities shape:", test_probs.shape)
print("Labels shape:", test_labels.shape)

Probabilities shape: (2898, 41)
Labels shape: (2898, 41)


In [ ]:

tuned_thresholds = (
    threshold_results.set_index("category")
    .loc[CATEGORIES, "threshold"]
    .to_numpy()
)

In [ ]:

test_preds_05 = (test_probs >= 0.5).astype(int)
test_preds_tuned = (test_probs >= tuned_thresholds).astype(int)


In [ ]:
print("Test macro F1 at 0.5:",
      round(f1_score(test_labels, test_preds_05,
                     average="macro", zero_division=0), 3))

print("Test macro F1 with tuned thresholds:",
      round(f1_score(test_labels, test_preds_tuned,
                     average="macro", zero_division=0), 3))

print("Test micro F1 at 0.5:",
      round(f1_score(test_labels, test_preds_05,
                     average="micro", zero_division=0), 3))

print("Test micro F1 with tuned thresholds:",
      round(f1_score(test_labels, test_preds_tuned,
                     average="micro", zero_division=0), 3))

Test macro F1 at 0.5: 0.642
Test macro F1 with tuned thresholds: 0.624
Test micro F1 at 0.5: 0.708
Test micro F1 with tuned thresholds: 0.661


The decision threshold makes it worse. We will go without the threshold tuning for the final legal-bert model.
